# tiny-world-model — Colab bootstrap

Run the cells top to bottom. Cell 2 pulls the latest code from GitHub, so after you
commit from Claude Code locally, re-run cell 2 and cell 3 to pick the changes up.

**One-time setup:** if the repo is private, add a GitHub token to Colab Secrets
(🔑 in the left sidebar) named `GITHUB_TOKEN`, and toggle *Notebook access* on.

## 1. Check the GPU

Runtime → Change runtime type → T4 GPU (or better) if this reports no GPU.

In [ ]:
!nvidia-smi

## 2. Get the code

Set `REPO` to your own GitHub path. This clones on the first run and pulls on later runs,
so re-running the cell is how you sync after committing locally.

In [ ]:
import os
import subprocess
from pathlib import Path

REPO   = "YOUR_GITHUB_USERNAME/tiny-world-model"   # <-- edit this
BRANCH = "main"
ROOT   = Path("/content/tiny-world-model")

# Private repo? Read a PAT from Colab Secrets. Public repo? This is skipped.
token = None
try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
except Exception:
    pass  # no secret set, or not on Colab

url = f"https://{token}@github.com/{REPO}.git" if token else f"https://github.com/{REPO}.git"

def run(cmd, cwd=None):
    """Run git, hiding the token from any output or traceback."""
    r = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True)
    out = (r.stdout + r.stderr)
    if token:
        out = out.replace(token, "***")
    print(out.strip())
    if r.returncode != 0:
        raise RuntimeError(f"git failed: {cmd[:2]}")

if ROOT.exists():
    run(["git", "remote", "set-url", "origin", url], cwd=ROOT)
    run(["git", "fetch", "origin", BRANCH], cwd=ROOT)
    run(["git", "reset", "--hard", f"origin/{BRANCH}"], cwd=ROOT)
else:
    run(["git", "clone", "--branch", BRANCH, url, str(ROOT)])

os.chdir(ROOT)
run(["git", "log", "-1", "--oneline"], cwd=ROOT)

## 3. Install the package

`-e` installs in editable mode, so a `git pull` in cell 2 is enough — no reinstall needed
unless `pyproject.toml` changed. Torch is left alone: Colab's preinstalled CUDA build is
the one you want.

In [ ]:
%pip install -q -e . -r requirements-colab.txt

Restart is only needed the first time, or after dependencies change:

In [ ]:
# Uncomment on the first install of a fresh runtime, then re-run cells 2 and 4.
# import IPython; IPython.Application.instance().kernel.do_shutdown(True)

## 4. Sanity check

`autoreload` means edited modules are re-imported on each cell run, so you can pull new
code and keep the same session.

In [ ]:
%load_ext autoreload
%autoreload 2

from tiny_world_model import device_info
print(device_info())

## 5. (Optional) Mount Drive for checkpoints

Colab runtimes are wiped when they disconnect. With Drive mounted, `artifacts_dir()`
writes checkpoints there automatically and training resumes across sessions.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from tiny_world_model.env import artifacts_dir
print(artifacts_dir())

## 6. Train

In [ ]:
from tiny_world_model.training import TrainConfig, train

cfg = TrainConfig(
    epochs=5,
    batch_size=128,
    n_rollouts=1024,
    run_name="colab-baseline",
)
metrics = train(cfg)
metrics["history"][-1]

## 7. Look at what it imagined

In [ ]:
import torch
import matplotlib.pyplot as plt
from tiny_world_model.data import BouncingBallDataset
from tiny_world_model.env import resolve_device
from tiny_world_model.models import TinyWorldModel
from tiny_world_model.training.train import _ckpt_path, load_checkpoint

device = resolve_device()
model = TinyWorldModel(context=cfg.context, width=cfg.width, depth=cfg.depth).to(device)
load_checkpoint(_ckpt_path(cfg.run_name), model, device=device)

ds = BouncingBallDataset(n_rollouts=4, steps=32, context=cfg.context, seed=123)
ctx, _ = ds[0]
frames = model.rollout(torch.from_numpy(ctx)[None].to(device), steps=12)[0].cpu()

fig, axes = plt.subplots(1, 12, figsize=(18, 2))
for i, ax in enumerate(axes):
    ax.imshow(frames[i], cmap="magma", vmin=0, vmax=1)
    ax.set_title(f"t+{i + 1}", fontsize=8)
    ax.axis("off")
plt.tight_layout()

## 8. Pushing changes back

Treat Colab as read-only for code: experiment in cells, then make the real edit locally in
Claude Code and commit. If you do need to commit from here, configure git first — and note
the token must have write access:

```
!git config user.email "you@example.com"
!git config user.name "Your Name"
```